# RecSys 02 – MovieLens Transformer Inference

This notebook loads the trained **Transformer-based sequential recommender** for MovieLens 20M and runs simple **top-K next-item predictions**.

Pipeline:
1. Load config and paths
2. Load preprocessed sequences (`movielens_sequences.csv`)
3. Rebuild the `SequenceDataset`
4. Load the trained `TransformerSeqModel` checkpoint
5. Given a user sequence, predict the top-K next movies (by `movieId`)


In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F

# Ensure src/ is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from recsys_transformer_seq.config import load_config, PROJECT_ROOT as CFG_ROOT
from recsys_transformer_seq.data.seq_dataset import SequenceDataset
from recsys_transformer_seq.models.transformer_seq import TransformerSeqModel

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CFG_ROOT:", CFG_ROOT)

PROJECT_ROOT: /Users/mackcesar/PycharmProjects/recsys-02-transformer-seq
CFG_ROOT: /Users/mackcesar/PycharmProjects/recsys-02-transformer-seq


## 1. Load config and sequences

We expect `data/processed/movielens_sequences.csv` to be created by:

```bash
export PYTHONPATH="$PWD/src"
python -m recsys_transformer_seq.data.movielens_preprocess
```

In [11]:
cfg = load_config()
max_len = cfg["data"]["max_sequence_length"]

seqs_path = PROJECT_ROOT / "data" / "processed" / "movielens_sequences.csv"
display(seqs_path)

dataset = SequenceDataset(seqs_path, max_len=max_len)
print(f"Num samples: {len(dataset):,}")
print(f"Num items (including padding): {dataset.num_items:,}")


PosixPath('/Users/mackcesar/PycharmProjects/recsys-02-transformer-seq/data/processed/movielens_sequences.csv')

Num samples: 19,861,770
Num items (including padding): 26,745


## 2. Load trained Transformer checkpoint

We expect `data/models/transformer_seq_movielens.pt` to be created by:

```bash
export PYTHONPATH="$PWD/src"
python -m recsys_transformer_seq.train_seq
```

In [5]:
from recsys_transformer_seq.data.seq_dataset import SequenceDataset
from recsys_transformer_seq.models.transformer_seq import TransformerSeqModel
ckpt_path = PROJECT_ROOT / "data" / "models" / "transformer_seq_movielens.pt"
print("Checkpoint path:", ckpt_path)

ckpt = torch.load(ckpt_path, map_location="cpu")
num_items = ckpt["num_items"]
max_len_ckpt = ckpt["max_len"]

print("num_items (from ckpt):", num_items)
print("max_len (from ckpt):", max_len_ckpt)

model_cfg = cfg["model"]
model = TransformerSeqModel(
    num_items=num_items,
    max_len=max_len_ckpt,
    d_model=model_cfg["embedding_dim"],
    nhead=model_cfg["num_heads"],
    num_layers=model_cfg["num_layers"],
    dropout=model_cfg["dropout"],
).eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
device


Checkpoint path: /Users/mackcesar/PycharmProjects/recsys-02-transformer-seq/data/models/transformer_seq_movielens.pt
num_items (from ckpt): 26745
max_len (from ckpt): 50


device(type='cpu')

## 3. Helper: top-K next-item recommendations

We’ll:
- pick a sample index from the dataset
- run the sequence through the model
- get the top-K item indices
- map them back to raw `movieId`s using `dataset.idx2item`.

In [12]:
def recommend_for_index(sample_idx: int, top_k: int = 10):
    """Return top-K next-item predictions for a given dataset index.

    Outputs a DataFrame with internal item index and raw movieId.
    """
    seq, target = dataset[sample_idx]
    seq_batch = seq.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(seq_batch)  # (1, num_items)
        probs = F.softmax(logits, dim=-1)
        topk = torch.topk(probs, k=top_k)

    top_indices = topk.indices[0].cpu().tolist()
    top_scores = topk.values[0].cpu().tolist()

    rows = []
    for idx, score in zip(top_indices, top_scores):
        movie_id = dataset.idx2item.get(idx, None)
        rows.append({
            "internal_idx": idx,
            "movieId": movie_id,
            "score": float(score),
        })

    return pd.DataFrame(rows)


## 4. Example: recommendations for a random user sequence

You can change `sample_idx` below or loop over several examples.

In [13]:
import random

sample_idx = random.randint(0, len(dataset) - 1)
print("Sample index:", sample_idx)

seq, target = dataset[sample_idx]
print("Sequence length (non-padding):", int((seq != 0).sum()))
print("Target internal item idx:", target.item())
print("Target raw movieId:", dataset.idx2item[target.item()])

recs = recommend_for_index(sample_idx, top_k=10)
recs

Sample index: 17324481
Sequence length (non-padding): 50
Target internal item idx: 5299
Target raw movieId: 5395


,internal_idx,movieId,score
0,24267,115921,0.000340
1,11435,49200,0.000280
2,4913,5008,0.000269
3,2627,2712,0.000261
4,9476,27778,0.000234
5,1480,1526,0.000228
6,8025,8707,0.000222
7,16395,82917,0.000219
8,22323,107649,0.000217
9,3820,3912,0.000210
